### 📖 Chú thích các hàm (API Documentation)

Dưới đây là giải thích chi tiết về vai trò và cách vận hành của từng hàm trong quy trình truyền nhận bảo mật tích hợp RSA & AES-GCM.

---

#### 1. `generate_and_hide_public_key(sound_ref, out_sound_ref)`
- **Vai trò:** Dành cho **Người nhận (Receiver)** khởi tạo kết nối an toàn.
- **Cách hoạt động:**
  - Tự động tạo cặp khóa RSA-2048 mới.
  - Lưu trữ khóa bí mật `private_key.pem` an toàn tại máy người nhận (trong thư mục `RSA_keys/`).
  - Chuyển đổi khóa công khai (Public Key) sang dạng chuỗi nhị phân (bits) và nhúng vào tệp âm thanh WAV sạch ban đầu bằng phương pháp LSB.
- **Tham số:**
  - `sound_ref` (`str`): Đường dẫn tệp âm thanh gốc sạch (vật chứa).
  - `out_sound_ref` (`str`): Đường dẫn lưu tệp âm thanh đã chứa Public Key đầu ra (mặc định: `'outputs/public_key_carrier.wav'`).

---

#### 2. `extract_public_key(sound_ref)`
- **Vai trò:** Dành cho **Người gửi (Sender)** trích xuất khóa công khai của người nhận từ file âm thanh nhận được.
- **Cách hoạt động:** Đọc các bits LSB tại các vị trí chẵn quy định ở đầu tệp tin âm thanh, chuyển đổi chuỗi nhị phân này ngược lại thành định dạng khóa DER của RSA.
- **Tham số:**
  - `sound_ref` (`str`): Đường dẫn tới tệp âm thanh chứa Public Key được gửi đến.
- **Trả về (`return`):** Chuỗi bytes của Public Key ở định dạng DER.

---

#### 3. `encoding_secure(carrier_key_sound, data_ref, output_sound_ref)`
- **Vai trò:** Dành cho **Người gửi (Sender)** thực hiện mã hóa thông tin và ẩn giấu dữ liệu vào file âm thanh.
- **Cách hoạt động:**
  - Gọi hàm `extract_public_key` để lấy Public Key của đối tác từ tệp `carrier_key_sound`.
  - Mã hóa dữ liệu gốc (bất kể định dạng văn bản hay hình ảnh) bằng khóa AES bí mật ngẫu nhiên (chế độ an toàn tuyệt đối AES-GCM).
  - Bảo vệ khóa AES (gồm cả `key`, `nonce`, và `tag` kiểm định) bằng cách mã hóa bất đối xứng qua RSA Public Key.
  - Nhúng chuỗi khóa RSA đã mã hóa và chuỗi nội dung mật mã (`ciphertext`) liên tiếp vào tệp âm thanh vật chứa thông qua LSB.
- **Tham số:**
  - `carrier_key_sound` (`str`): Đường dẫn tệp âm thanh chứa Public Key nhận được từ người nhận.
  - `data_ref` (`str`): Đường dẫn tới tệp tin chứa dữ liệu gốc cần mã hóa và gửi đi (ví dụ: `text.txt` hoặc `image.jpg`).
  - `output_sound_ref` (`str`): Đường dẫn lưu tệp âm thanh mật mã hoàn chỉnh gửi ngược lại (mặc định: `'outputs/encrypted_sound.wav'`).

---

#### 4. `decoding_secure(encrypted_sound_ref, private_key_path, out_data_path)`
- **Vai trò:** Dành cho **Người nhận (Receiver)** thực hiện trích xuất và giải mã khi nhận được tệp âm thanh mật.
- **Cách hoạt động:**
  - Đọc tệp âm thanh mật mã nhận được.
  - Trích xuất 2048 bits LSB đầu tiên để lấy chuỗi khóa RSA đã bị mã hóa.
  - Sử dụng **Private Key** cục bộ của mình để giải mã chuỗi khóa trên nhằm khôi phục bộ 3 tham số giải mã AES (`key`, `nonce`, `tag`).
  - Đọc header để xác định độ dài và trích xuất phần nội dung `ciphertext` tiếp theo.
  - Giải mã AES-GCM và ghi kết quả phục hồi ra file đích.
- **Tham số:**
  - `encrypted_sound_ref` (`str`): Đường dẫn tệp âm thanh chứa tin mật nhận từ phía gửi.
  - `private_key_path` (`str`): Đường dẫn tệp khóa Private Key lưu trữ trên máy người nhận (mặc định: `'RSA_keys/private_key.pem'`).
  - `out_data_path` (`str`): Đường dẫn để xuất file dữ liệu gốc sau khi khôi phục thành công (mặc định: `'outputs/retrieved_file.bin'`).

In [ ]:
import wave as w
import os
from Crypto.Cipher import AES
from Crypto.PublicKey import RSA
from Crypto.Cipher import PKCS1_OAEP
from Crypto.Random import get_random_bytes

# Helper: Tạo thư mục cần thiết
os.makedirs('outputs', exist_ok=True)
os.makedirs('RSA_keys', exist_ok=True)

# --- Chuyển đổi dữ liệu và bits ---
def bytes_to_bits(data_bytes):
    return ''.join(format(b, '08b') for b in data_bytes)

def bits_to_bytes(bits_str):
    return bytes(int(bits_str[i:i+8], 2) for i in range(0, len(bits_str), 8))

# --- Đọc/Ghi dữ liệu nguồn ---
def file_to_bytes(path):
    with open(path, 'rb') as f:
        return f.read()

# --- 1. Sinh Khóa RSA & Nhúng Public Key vào Carrier WAV (Dành cho Người Nhận) ---
def generate_and_hide_public_key(sound_ref, out_sound_ref='outputs/public_key_carrier.wav'):
    # Tạo cặp khóa RSA
    key = RSA.generate(2048)
    private_key = key.export_key(format='DER')
    public_key = key.publickey().export_key(format='DER')
    
    # Lưu Private Key cục bộ (không gửi đi)
    with open('RSA_keys/private_key.pem', 'wb') as f:
        f.write(private_key)
    print("Đã lưu Private Key cục bộ thành công tại 'RSA_keys/private_key.pem'")

    # Đọc file âm thanh làm vật chứa
    sound = w.open(sound_ref, 'rb')
    params = sound.getparams()
    sound_data = bytearray(sound.readframes(sound.getnframes()))
    sound.close()

    # Chuyển Public Key thành bits (Độ dài DER của RSA-2048 public key thường khoảng 294 bytes = 2352 bits)
    pub_bits = bytes_to_bits(public_key)
    pub_len = len(pub_bits)

    # Giấu độ dài Public Key (32 bits) ở 32 frames đầu tiên
    len_bits = format(pub_len, '032b')
    idx = 0
    for bit in len_bits:
        sound_data[idx] = (sound_data[idx] & 254) | int(bit)
        idx += 2

    # Giấu Public Key vào các frames tiếp theo
    for bit in pub_bits:
        sound_data[idx] = (sound_data[idx] & 254) | int(bit)
        idx += 2

    # Lưu file âm thanh mang Public Key
    out_sound = w.open(out_sound_ref, 'wb')
    out_sound.setparams(params)
    out_sound.writeframes(sound_data)
    out_sound.close()
    print(f"Đã tạo thành công file âm thanh chứa Public Key tại '{out_sound_ref}'")

# --- 2. Trích xuất Public Key từ Carrier WAV (Dành cho Người Gửi) ---
def extract_public_key(sound_ref):
    sound = w.open(sound_ref, 'rb')
    sound_data = bytearray(sound.readframes(sound.getnframes()))
    sound.close()

    # Lấy độ dài Public Key (32 bits đầu, bước nhảy 2)
    len_bits = ''.join(str(sound_data[i] & 1) for i in range(0, 64, 2))
    pub_len = int(len_bits, 2)

    # Lấy chuỗi bits của Public Key
    pub_bits = ''.join(str(sound_data[i] & 1) for i in range(64, 64 + pub_len * 2, 2))
    pub_bytes = bits_to_bytes(pub_bits)
    return pub_bytes

# --- 3. Mã hóa & Giấu Tin Mật (Dành cho Người Gửi) ---
def encoding_secure(carrier_key_sound, data_ref, output_sound_ref='outputs/encrypted_sound.wav'):
    # Bước A: Trích xuất Public Key của người nhận
    public_key_der = extract_public_key(carrier_key_sound)
    public_key = RSA.importKey(public_key_der)

    # Bước B: Đọc dữ liệu mật cần gửi
    plain_bytes = file_to_bytes(data_ref)

    # Bước C: Mã hóa dữ liệu bằng AES-GCM
    aes_key = get_random_bytes(16)
    cipher_aes = AES.new(aes_key, AES.MODE_GCM)
    ciphertext, tag = cipher_aes.encrypt_and_digest(plain_bytes)
    nonce = cipher_aes.nonce

    # Bước D: Ghép key + nonce + tag (48 bytes) rồi mã hóa bằng RSA Public Key
    combined_keys = aes_key + nonce + tag
    cipher_rsa = PKCS1_OAEP.new(public_key)
    rsa_encrypted_bytes = cipher_rsa.encrypt(combined_keys)

    # Chuyển đổi toàn bộ thành bits
    rsa_encrypted_bits = bytes_to_bits(rsa_encrypted_bytes) # 256 bytes = 2048 bits
    ciphertext_bits = bytes_to_bits(ciphertext)
    
    # Đọc file âm thanh nguồn để nhúng tin mật
    sound = w.open(carrier_key_sound, 'rb')
    params = sound.getparams()
    sound_data = bytearray(sound.readframes(sound.getnframes()))
    sound.close()

    # Vị trí nhúng an toàn bắt đầu sau khu vực Public Key (ví dụ: bắt đầu từ index 10000 để tránh đè lên Public key cũ nếu muốn bảo toàn, hoặc đè trực tiếp)
    # Quy ước cấu trúc nhúng tin mật ở đầu file âm thanh đầu ra:
    # - 2048 bits đầu (0 -> 4096): RSA encrypted keys
    # - 32 bits tiếp theo (4096 -> 4160): Độ dài ciphertext
    # - Phần tiếp theo: Toàn bộ ciphertext bits
    
    idx = 0
    # 1. Nhúng RSA encrypted keys (2048 bits)
    for bit in rsa_encrypted_bits:
        sound_data[idx] = (sound_data[idx] & 254) | int(bit)
        idx += 2

    # 2. Nhúng độ dài ciphertext (32 bits)
    size_bits = format(len(ciphertext_bits), '032b')
    for bit in size_bits:
        sound_data[idx] = (sound_data[idx] & 254) | int(bit)
        idx += 2

    # 3. Nhúng Ciphertext bits
    if len(ciphertext_bits) > (len(sound_data) - idx) // 2:
        raise ValueError("File âm thanh quá ngắn để nhúng toàn bộ dữ liệu mã hóa!")
        
    for bit in ciphertext_bits:
        sound_data[idx] = (sound_data[idx] & 254) | int(bit)
        idx += 2

    # Ghi file kết quả
    out_sound = w.open(output_sound_ref, 'wb')
    out_sound.setparams(params)
    out_sound.writeframes(sound_data)
    out_sound.close()
    print(f"Đã tạo thành công file âm thanh mật mã tại '{output_sound_ref}'")

# --- 4. Giải Mã & Khôi Phục Dữ Liệu Gốc (Dành cho Người Nhận) ---
def decoding_secure(encrypted_sound_ref, private_key_path='RSA_keys/private_key.pem', out_data_path='outputs/retrieved_file.bin'):
    # Đọc Private Key cục bộ
    with open(private_key_path, 'rb') as f:
        private_key_der = f.read()
    private_key = RSA.importKey(private_key_der)

    # Đọc file âm thanh
    sound = w.open(encrypted_sound_ref, 'rb')
    sound_data = bytearray(sound.readframes(sound.getnframes()))
    sound.close()

    idx = 0
    # 1. Trích xuất RSA Encrypted Keys (2048 bits = 256 bytes)
    rsa_bits = ''.join(str(sound_data[i] & 1) for i in range(0, 4096, 2))
    rsa_encrypted_bytes = bits_to_bytes(rsa_bits)
    idx = 4096

    # Giải mã RSA để lấy aes_key, nonce, tag
    cipher_rsa = PKCS1_OAEP.new(private_key)
    combined_keys = cipher_rsa.decrypt(rsa_encrypted_bytes)
    
    aes_key = combined_keys[0:16]
    nonce = combined_keys[16:32]
    tag = combined_keys[32:48]

    # 2. Trích xuất độ dài Ciphertext (32 bits)
    size_bits = ''.join(str(sound_data[i] & 1) for i in range(idx, idx + 64, 2))
    ciphertext_len = int(size_bits, 2)
    idx += 64

    # 3. Trích xuất Ciphertext
    ciphertext_bits = ''.join(str(sound_data[i] & 1) for i in range(idx, idx + ciphertext_len * 2, 2))
    ciphertext = bits_to_bytes(ciphertext_bits)

    # 4. Giải mã AES-GCM
    cipher_aes = AES.new(aes_key, AES.MODE_GCM, nonce=nonce)
    decrypted_bytes = cipher_aes.decrypt_and_verify(ciphertext, tag)

    # Ghi ra file kết quả
    with open(out_data_path, 'wb') as f:
        f.write(decrypted_bytes)
    print(f"Đã giải mã thành công! File đầu ra lưu tại: '{out_data_path}'")